# 2. MC Dropout, and splitting the uncertainty in two

Two things happen in this notebook.

First, the cheapest possible uncertainty method: train with dropout as usual, then
leave dropout switched on at test time and push each input through the network `T`
times. The `T` answers are approximate samples from a posterior over weights, at
no extra training cost.

Second, the split that the rest of the project depends on. Once `S > 1` you can
separate the noise the model reports from the disagreement between sampled
networks, and only the second one shrinks when you collect more data.

You write two functions: `predict_mc_dropout` and `decompose_variance`.

About 1.5 hours.

## 2.1 Dropout as an approximate posterior

Dropout multiplies each weight matrix by a random 0/1 mask during training. Gal
and Ghahramani's observation is that this is already variational inference: the
masked weights are a sample from a variational distribution `q(W)`, and the usual
dropout training objective is the ELBO for that `q` up to a constant. So the
samples you need are available for free, provided you keep the masks switched on
when predicting.

What that buys and what it does not: the width of `q` here is set by the dropout
rate `p` and by the trained weights, and it does not depend on how much data you
have. A posterior should contract as data accumulates. This one does not, so
treat MC Dropout as a cheap source of function-space jitter whose usefulness is
an empirical question, not as an approximation that improves with `N`.

The derivation is in `ASSIGNMENT.md`; `THEORY.md` has it in full.

In [ ]:
import sys
import time

sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import numpy as np
import torch

from bdl.data import load_track, make_toy1d, toy1d_grid
from bdl.metrics import evaluate, interval_coverage, results_table
from bdl.models import (
    build_model,
    default_loss,
    enable_dropout,
    fit,
    gaussian_head,
    set_seed,
    track_hparams,
)
from bdl.plots import plot_band, plot_reliability, plot_uncertainty_vs_x
from bdl.store import load_run, save_run

%matplotlib inline

ds = make_toy1d()
grid = toy1d_grid()


# You wrote this in notebook 01. It is repeated here so this notebook stands on
# its own; nothing is imported from another notebook anywhere in the project.
def calibration_error(mu, sigma, y, n_bins=15):
    levels = torch.linspace(0.0, 1.0, n_bins + 2)[1:-1]
    gaps = [abs(interval_coverage(mu, sigma, y, float(q)) - float(q)) for q in levels]
    return float(np.mean(gaps))

## 2.2 Training is unchanged

Nothing about training is special in MC Dropout, which is the point of the
method. The only difference from notebook 01 is `dropout=P_DROP` when the model
is built.

In [ ]:
P_DROP = 0.1
T = 50  # forward passes at test time

set_seed(0)
model = build_model(ds, hidden=(64, 64), dropout=P_DROP)
history = fit(
    model, ds.x_train, ds.y_train, loss_fn=default_loss(ds), epochs=400, lr=1e-2, seed=0
)
print(f"final training loss: {history[-1]:.4f}")
print(model.net[2])  # the dropout layer sitting after the first activation

## 2.3 Predicting with dropout left on

Write `predict_mc_dropout`. Three steps:

1. put the model in eval mode, then call `enable_dropout(model)`. That switches
   the dropout layers, and only those, back into training mode;
2. run `T` forward passes, collecting `mu` and `sigma` from `gaussian_head` each
   time;
3. stack them into two arrays of shape `[T, N]`.

Use `enable_dropout(model)` rather than `model.train()`. `model.train()` also puts
batch normalisation into training mode, which makes it normalise with statistics
of the current batch, so your prediction for one test point would depend on which
other test points happen to sit beside it. This MLP has no batch normalisation, so
here the two behave the same and the bug would stay invisible until it mattered.

Wrap the function in `@torch.no_grad()`: no gradients are needed and it is
roughly twice as fast.

In [ ]:
@torch.no_grad()
def predict_mc_dropout(model, x, n_samples=50):
    """T stochastic forward passes with dropout active. Returns mu, sigma of shape [T, N]."""
    # ---- TODO ------------------------------------------------------------
    # 1. model.eval(), then enable_dropout(model)
    # 2. n_samples forward passes, each giving mu, sigma = gaussian_head(model(x))
    # 3. torch.stack the two lists -> [n_samples, N]
    raise NotImplementedError
    # ----------------------------------------------------------------------

In [ ]:
# ---- check your work -------------------------------------------------------
mu_t, sd_t = predict_mc_dropout(model, ds.x_test, n_samples=T)
assert mu_t.shape == sd_t.shape == (T, len(ds.x_test)), (mu_t.shape, sd_t.shape)
assert torch.all(sd_t > 0), "sigma must be positive"

# The samples must actually differ. If they do not, dropout is off.
spread = float(mu_t.std(0).mean())
assert spread > 1e-4, f"the T samples are identical (spread {spread:.2e}) -- is dropout on?"

# And they must differ *because of dropout*, not by accident: with dropout off,
# repeated passes are identical.
model.eval()
with torch.no_grad():
    a, _ = gaussian_head(model(ds.x_test))
    b, _ = gaussian_head(model(ds.x_test))
assert torch.allclose(a, b), "eval mode should be deterministic"

print(f"OK   {T} samples, mean spread between them {spread:.4f}")

## 2.4 The two uncertainties

The predictive distribution is the mixture `(1/S) sum_s N(mu_s, sigma_s^2)`. Its
variance splits exactly in two. Writing `W` for the weights,

    Var[y] = E_W[ Var[y | W] ]  +  Var_W[ E[y | W] ]
           = E_W[ sigma^2(x) ]  +  Var_W[ mu(x) ]
             \_____________/       \___________/
               aleatoric             epistemic

Average the per-sample noise variances to get the aleatoric part; take the
variance of the per-sample means across samples to get the epistemic part. This is
an identity, not an approximation, so your `total` must equal
`aleatoric + epistemic` to machine precision. The way to guarantee that is to
return their sum rather than computing the total separately.

Use the population variance, `unbiased=False`. With `S = 1` it gives 0, which is
the right answer for a single network, where the `1/(S-1)` version would give
`NaN`.

One consequence worth remembering: the epistemic term is a variance of means, so
it has units of `y` squared and is not comparable between methods. Compare shapes
and ratios, not magnitudes.

In [ ]:
def decompose_variance(mu, sigma):
    """Split the predictive variance. mu, sigma: [S, N]. Returns three [N] tensors.

    Returns (total, aleatoric, epistemic).
    """
    # ---- TODO ------------------------------------------------------------
    # aleatoric: average the per-sample noise variances over the sample axis
    # epistemic: variance of the per-sample means over the sample axis
    #            (unbiased=False, so S = 1 gives 0 rather than NaN)
    # total:     return their sum, so the identity holds exactly
    raise NotImplementedError
    # ----------------------------------------------------------------------

In [ ]:
# ---- check your work -------------------------------------------------------
torch.manual_seed(0)

# 1. The identity holds exactly.
mu_r, sd_r = torch.randn(7, 40), torch.rand(7, 40) + 0.1
tot, ale, epi = decompose_variance(mu_r, sd_r)
assert torch.allclose(tot, ale + epi, atol=1e-6), "total must equal aleatoric + epistemic"

# 2. The total is the variance of the mixture. Checked by drawing from the
#    mixture instead of by reusing the formula.
mu_x = torch.tensor([[-2.0, 0.5], [1.0, 0.5], [3.0, 0.5]])  # S=3, N=2
sd_x = torch.tensor([[0.5, 2.0], [1.5, 2.0], [0.5, 2.0]])
tot_x, _, _ = decompose_variance(mu_x, sd_x)
rng = np.random.default_rng(0)
for n in range(2):
    comp = rng.integers(0, 3, size=400_000)
    draws = rng.normal(mu_x[comp, n].numpy(), sd_x[comp, n].numpy())
    assert abs(float(tot_x[n]) / draws.var() - 1) < 0.02, (float(tot_x[n]), draws.var())

# 3. One sample means no disagreement; identical samples likewise.
_, ale1, epi1 = decompose_variance(torch.randn(1, 10), torch.rand(1, 10) + 0.5)
assert float(epi1.abs().max()) < 1e-10
_, _, epi_same = decompose_variance(torch.full((5, 8), 1.2), torch.full((5, 8), 0.7))
assert float(epi_same.abs().max()) < 1e-10

print("OK   identity exact, total matches the sampled mixture, S=1 gives zero")

## 2.5 What the split looks like

Two figures. In the first, the inner band is aleatoric and the outer band is the
total, so the gap between them is the epistemic part. Compare it with the same
figure in notebook 01, where the two coincided.

In the second, the epistemic standard deviation against `x`, with the regions
that have no training data shaded. This is the direct test of whether the model
knows what it does not know.

In [ ]:
mu_g, sd_g = predict_mc_dropout(model, grid, n_samples=T)
tot_g, ale_g, epi_g = decompose_variance(mu_g, sd_g)

fig, ax = plt.subplots(figsize=(9, 5))
plot_band(ax, grid, mu_g.mean(0), ale_g.sqrt(), tot_g.sqrt(), ds=ds, title=f"MC Dropout, T={T}")
ax.legend(fontsize=8, loc="upper left")
plt.tight_layout()

In [ ]:
plot_uncertainty_vs_x({"mc_dropout": epi_g.sqrt()}, grid, ds);

## 2.6 Scored, next to the baseline

`evaluate` assembles the whole metrics row. It needs the two functions you wrote,
which is why it takes them as arguments: every method from here on is scored by
exactly the same code.

In [ ]:
mu_id, sd_id = predict_mc_dropout(model, ds.x_test, n_samples=T)
mu_ood, sd_ood = predict_mc_dropout(model, ds.x_ood, n_samples=T)

metrics = evaluate(
    (mu_id, sd_id),
    ds.y_test,
    (mu_ood, sd_ood),
    ds.y_ood,
    task=ds.task,
    decompose=decompose_variance,
    calibration=calibration_error,
)
metrics["train_s"] = 0.0

baseline = load_run("toy1d", "deterministic")["metrics"]
print(results_table({"deterministic": baseline, "mc_dropout": metrics}))

## 2.7 Things to look at before moving on

* Does the epistemic standard deviation peak inside the gap and outside the
  training range? If it is flat, your samples are not really different.
* The OOD AUROC is no longer 0.5. There is now a per-point score that varies, so
  the rank statistic has something to work with.
* Compare the shifted NLL with the baseline's. The mean prediction in the gap is
  no better than before; what changed is that the model widened its interval
  there, and the NLL rewards that.
* Coverage at 95% may now be above 0.95. Underconfidence is also miscalibration,
  and `ece` counts it.

## 2.8 Track C only: the same split for classification

Skip this section if you are on Track A or B.

For a classifier there is no `sigma` to average, so the split is
information-theoretic instead:

    H[ E_W p(y|x,W) ]  =  I(y; W | x)  +  E_W[ H(p(y|x,W)) ]
    \_______________/     \__________/     \________________/
      total                epistemic        aleatoric
      (entropy of the      (mutual          (average entropy of
       mean softmax)        information)     the individual softmaxes)

Read it as: total confusion equals disagreement between plausible models plus the
confusion they all share. Compute the total and the aleatoric part directly and
obtain the epistemic part as the difference, so the identity holds exactly. Work
in nats and clamp the probabilities away from zero before taking a log.

Track C also needs a calibration error for classification: bin the predictions by
the top-class probability and compare the mean confidence in each bin with the
accuracy in that bin,

    ECE = sum_b (n_b / N) * | accuracy(b) - confidence(b) |

with `n_bins` equal-width bins over `[0, 1]` and empty bins contributing nothing.

In [ ]:
def decompose_entropy(probs):
    """Split predictive entropy. probs: [S, N, K]. Returns (total, aleatoric, epistemic)."""
    # ---- TODO (Track C only) ---------------------------------------------
    # total     : entropy of the mean softmax vector
    # aleatoric : mean over samples of the entropy of each softmax vector
    # epistemic : total - aleatoric
    # natural log, and clamp probabilities to at least 1e-12 before any log
    raise NotImplementedError
    # ----------------------------------------------------------------------


def calibration_error_probs(probs, y, n_bins=15):
    """Binned confidence-vs-accuracy gap. probs: [S, N, K]."""
    # ---- TODO (Track C only) ---------------------------------------------
    # confidence = top-class probability of the mean softmax
    # bin over [0, 1] in n_bins equal-width bins, skip empty bins, and return
    # sum_b (n_b / N) * |accuracy(b) - mean confidence(b)|
    raise NotImplementedError
    # ----------------------------------------------------------------------

In [ ]:
# ---- check your work (Track C only) ----------------------------------------
# On Tracks A and B these two functions are left unimplemented, so this cell
# prints a note and moves on instead of failing.
import math

try:
    # Two networks, each certain, and certain of a different class: every
    # individual softmax has zero entropy, and the mean is (0.5, 0.5).
    probs = torch.tensor([[[1.0, 0.0]], [[0.0, 1.0]]])
    tot, ale, epi = decompose_entropy(probs)
    assert abs(float(tot) - math.log(2)) < 1e-5, float(tot)
    assert abs(float(ale)) < 1e-5, float(ale)
    assert abs(float(epi) - math.log(2)) < 1e-5, float(epi)

    # Two networks that agree the answer is a coin flip: nothing epistemic.
    probs = torch.tensor([[[0.5, 0.5]], [[0.5, 0.5]]])
    tot, ale, epi = decompose_entropy(probs)
    assert abs(float(ale) - math.log(2)) < 1e-5 and abs(float(epi)) < 1e-5

    # The identity holds, and the mutual information is never negative.
    p_rand = torch.softmax(torch.randn(6, 30, 4), dim=-1)
    tot, ale, epi = decompose_entropy(p_rand)
    assert torch.allclose(tot, ale + epi, atol=1e-6)
    assert float(epi.min()) > -1e-6

    # A classifier that always claims 100% and is right half the time: ECE 0.5.
    n_c = 1000
    certain = torch.zeros(1, n_c, 2)
    certain[0, :, 0] = 1.0
    y_half = torch.tensor([0, 1] * (n_c // 2))
    assert abs(calibration_error_probs(certain, y_half) - 0.5) < 1e-6

    print("OK   entropy decomposition and classification ECE both correct")
except NotImplementedError:
    print("skipped: this section is for Track C only")

## 2.9 Your own track

Same code, your dataset. The only branch is which decomposition and which
calibration error to hand to `evaluate`.

In [ ]:
TRACK = "A"  # <-- keep the same track for the whole project

ds_track = load_track(TRACK)
hp = track_hparams(TRACK, "mc_dropout")
print(ds_track)
print("hyperparameters:", hp)

t0 = time.perf_counter()
set_seed(0)
model_track = build_model(ds_track, hidden=(64, 64), dropout=P_DROP)
fit(
    model_track,
    ds_track.x_train,
    ds_track.y_train,
    loss_fn=default_loss(ds_track),
    epochs=int(hp["epochs"]),
    lr=hp["lr"],
    seed=0,
)
train_s = time.perf_counter() - t0
print(f"trained in {train_s:.1f}s")

In [ ]:
@torch.no_grad()
def predict_probs_mc_dropout(model, x, n_samples=50):
    """The classification version: T sampled softmax vectors, shape [T, N, K]."""
    model.eval()
    enable_dropout(model)
    return torch.stack([torch.softmax(model(x), dim=-1) for _ in range(n_samples)])


if ds_track.task == "regression":
    pred_id = predict_mc_dropout(model_track, ds_track.x_test, n_samples=T)
    pred_ood = predict_mc_dropout(model_track, ds_track.x_ood, n_samples=T)
    row = evaluate(
    pred_id,
    ds_track.y_test,
    pred_ood,
    ds_track.y_ood,
    task=ds_track.task,
    decompose=decompose_variance,
    calibration=calibration_error,
    )
else:
    pred_id = (predict_probs_mc_dropout(model_track, ds_track.x_test, n_samples=T),)
    pred_ood = (predict_probs_mc_dropout(model_track, ds_track.x_ood, n_samples=T),)
    row = evaluate(
    pred_id,
    ds_track.y_test,
    pred_ood,
    None,
    task=ds_track.task,
    decompose=decompose_entropy,
    calibration=calibration_error_probs,
    )

row["train_s"] = round(train_s, 2)
print(results_table({"deterministic": load_run(TRACK, "deterministic")["metrics"], "mc_dropout": row}))
save_run(TRACK, "mc_dropout", row)

In [ ]:
levels = np.linspace(0.05, 0.95, 12)
coverage = [interval_coverage(mu_id, sd_id, ds.y_test, float(q)) for q in levels]
plot_reliability({"mc_dropout": (levels, coverage)});

save_run(
    "toy1d",
    "mc_dropout",
    metrics,
    mean=mu_g.mean(0),
    sd_aleatoric=ale_g.sqrt(),
    sd_total=tot_g.sqrt(),
    epistemic_std=epi_g.sqrt(),
    levels=levels,
    coverage=np.array(coverage),
)

## Done when

* both check cells print `OK`;
* the epistemic band is visibly wider inside the gap than on the training data;
* `results/toy1d/mc_dropout.json` and `results/<your track>/mc_dropout.json` exist.

Next: notebook 03 gets its samples a different way, by training five networks
instead of one.